# Discretization workflows with FloPy

In this notebook we'll cover how to build Structured and Unstructured Meshes (Grids) with FloPy for use in MODFLOW models.

This notebook will start with generating a simple Structured Grid and we will use this grid object to begin developing Unstructured Meshes. The Unstructured Meshes we'll generate include:

   - Structured Grids (DIS): Standardard rectilinear discretization
   - Local Grid Refinement (LGR): A locally refined mesh that can be subsetted into a second model within a MODFLOW simulation
   - Quadtree Mesh: A mesh with a four fold (quad) refinement for each grid cell in a given refinement area
   - Triangular Mesh: A triangular mesh generated using Delaunay triangulation
   - Voronoi Mesh: A mesh generated from the connected centers of the Delaunay triangulation circumcircles

In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

from flopy.plot import PlotMapView
from flopy.discretization import StructuredGrid, VertexGrid
from flopy.utils.lgrutil import Lgr
from flopy.utils.gridgen import Gridgen
from flopy.utils.triangle import Triangle
from flopy.utils.voronoi import VoronoiGrid
from flopy.utils.rasters import Raster
from flopy.utils.gridintersect import GridIntersect

import warnings
warnings.simplefilter("ignore", DeprecationWarning)

In [ ]:
data_ws = Path("../data/sagehen_geospatial")

## Creating a structured (rectilinear) grid

This activity will start by creating a simple sturctured (rectilinear) grid from a 1/3 arc second Digital Elevation Model of the Sagehen Creek watershed as an illustration of how we can create an initial mesh that can be used to develop a model from.

Load the Digital Elevation Model using FloPy and visualize it

In [ ]:
dem = data_ws / "dem.img"

In [ ]:
rstr = Raster.load(dem)
rstr.plot();

Get the DEM boundaries

In [ ]:
xmin, xmax, ymin, ymax = rstr.bounds
print(xmin, xmax)

And begin developing the initial parameters for building our discretization

In [ ]:
dx = 100
dy = 150
lx = xmax - xmin
ly = ymax - ymin
nlay = 1
nrow = int(np.floor(ly / dy))
ncol = int(np.floor(lx / dx))
delc = np.full((nrow,), dy)
delr = np.full((ncol,), dx)

Finally create a fake top, bottom, and idomain array for the model (for now). These will be updated in a subsequent step.

In [ ]:
top = np.ones((nrow, ncol))
botm = np.zeros((nlay, nrow, ncol))
idomain = np.ones((nlay, nrow, ncol), dtype=int)

And now we can generate a `StructuredGrid` object that can be used for developing model inputs


In [ ]:
# live code
crs = "EPSG:26911"
sgrid = StructuredGrid(
    delc=delc,
    delr=delr,
    top=top,
    botm=botm,
    idomain=idomain,
    xoff=xmin,
    yoff=ymin,
    crs=crs
)

We can plot our `SturcturedGrid` object over the existing raster data to check that it is correctly orientied geographic in space


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
rstr.plot(ax=ax)
pmv = PlotMapView(modelgrid=sgrid, ax=ax)
pmv.plot_grid(lw=0.5);

We can also convert the grid to geodataframe and write it to shapefile for use with external tools like QGIS.

### Resampling some data to finish creating our initial grid

Once we have an initial grid, raster and vector data can be resampled or intersected with the grid to begin building a model domain. Although the focus of this notebook is to generate different types of meshes for modeling, the resampling and intersection process in this notebook can be applied many different data types and be used to generate boundary conditions within a MODFLOW model as we'll explore in a later notebook.

#### Raster resampling
We'll use FloPy's built in raster resampling method to generate values for land surface elevation (`top`) using the existing `StructuredGrid` object. For more information on raster resampling, please see this [example](https://flopy.readthedocs.io/en/latest/Notebooks/raster_intersection_example.html)

In [ ]:
top = rstr.resample_to_grid(
    sgrid,
    rstr.bands[0],
    method="min",
    extrapolate_edges=True
)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
pmv = PlotMapView(modelgrid=sgrid)
pc = pmv.plot_array(top)
pmv.plot_grid(lw=0.5)

#### Vector (shapefile) intersection

We can also intersect lines, points, and polygons with an existing modelgrid object. In this example, we'll load up the watershed boundary and get the active and inactive areas of the grid. For more information on intersecting shapefiles with a modelgrid/model see this [example](https://flopy.readthedocs.io/en/latest/Notebooks/grid_intersection_example.html)

In [ ]:
nhd_bounds = data_ws / "sagehen_watershed.shp"
basindf = gpd.read_file(nhd_bounds)
basindf.plot();

Using `GridIntersect` to intersect the basin with the modelgrid. More on this in a following notebook.

In [ ]:
ix = GridIntersect(sgrid)
result = ix.intersect(basindf.geometry.values[0], contains_centroid=True)
rowcol = result["cellids"]
rowcol

In [ ]:
row, col = list(zip(*rowcol))

Buidling a new idomain from the intersection

In [ ]:
idomain = np.zeros((nlay, nrow, ncol), dtype=int)
idomain[:, row, col] = 1

Putting it all together to create a complete Structured Grid discretization

In [ ]:
sgrid = StructuredGrid(
    delc=delc,
    delr=delr,
    top=top,
    botm=sgrid.botm,
    idomain=idomain,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    xoff=sgrid.xoffset,
    yoff=sgrid.yoffset,
    crs=sgrid.crs
)
sgrid

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
pmv = PlotMapView(modelgrid=sgrid)
pc = pmv.plot_array(sgrid.top)
ib = pmv.plot_inactive()
pmv.plot_grid(lw=0.5)
plt.colorbar(pc, shrink=0.7);

## Quadtree Mesh

A quadtree mesh is an unstructured refinement to a rectilinear modelgrid. In a quadtree mesh each node is split into 4 child nodes per level of refinement. The figure below is from [Lien and others, 2019](https://pubs.usgs.gov/of/2014/1109/pdf/ofr2014-1109.pdf) and shows multiple levels of recursive refinement.

![Quadmesh.png](../data/Quadmesh.png)

For this example, we will be using GRIDGEN and the FloPy utility `Gridgen` to create quadtree refinement around the streams in this watershed.

In [ ]:
# create a directory to build the gridgen files
gridgen_dir = data_ws / "gridgen"
gridgen_dir.mkdir(exist_ok=True)

In [ ]:
# load up the streams we'll use as refinement features
nhd_streams = data_ws / "sagehen_nhd.shp"
gdf = gpd.read_file(nhd_streams)
gdf.plot();

The `Gridgen` class a number of input parameters and further documentation/examples of the class can be found [here](https://flopy.readthedocs.io/en/latest/Notebooks/gridgen_example.html). For our example, we will be refining the area around the stream channels.

In [ ]:
g = Gridgen(sgrid, model_ws=gridgen_dir)
for geom in gdf.geometry.values:
    xy = list(zip(*geom.coords.xy))
    g.add_refinement_features([xy], "line", 2, range(nlay))

In [ ]:
g.build()

After gridgen builds the mesh, it can be used to create a `VertexGrid` (or DISV) that represents the basin

In [ ]:
gridprops = g.get_gridprops_disv()
gridprops.pop("nvert")

quadgrid = VertexGrid(**gridprops)
quadgrid.plot();

Resample top elevations to the quadtree mesh and intersect the basin boundary and stream vectors

In [ ]:
# create the top array
top = rstr.resample_to_grid(
    quadgrid,
    band=rstr.bands[0],
    method="min"
)

# sneaky little trick so we don't need to rebuild the grid
quadgrid._top = top

In [ ]:
# create the idomain array
idomain = np.zeros(quadgrid.shape, dtype=int)
ix = GridIntersect(quadgrid)
nodes = ix.intersect(basindf.geometry.values[0], contains_centroid=True)["cellids"]
idomain[:, list(nodes)] = 1

# once again, a sneaky little trick to update the idomain array inplace
quadgrid._idomain = idomain

Plot the quadtree mesh with top elevations and overlay the stream array

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

pmv = PlotMapView(modelgrid=quadgrid)
pc = pmv.plot_array(quadgrid.top)
pmv.plot_inactive()
pmv.plot_grid(lw=0.4)
gdf.plot(ax=ax, lw=0.8, color="red")
plt.colorbar(pc, shrink=0.7);

## Voronoi Mesh

Voronoi meshes are a little more involved to build than the previous two types of meshes, because it's a multistep process that begins with building a triangular mesh and then creating the voronoi mesh from the triangular mesh. To begin, let's start with the triangular mesh


#### Generating a Triangular mesh first
An unstructured/vertex based triangular mesh can be generated using the code "triangle"([Shewchuk, 2002](https://www.sciencedirect.com/science/article/pii/S0925772101000475)). And the FloPy utility, `Triangle` that creates inputs for and translates the outputs from "triangle".

For this example, we will be using triangle to generate a mesh that adds refinement around the streams.

In [ ]:
tri_dir = data_ws / "triangle"
tri_dir.mkdir(exist_ok=True)

For this example, we will use the watershed boundary and create a dissovled polygon of stream segments to generate our triangulation regions.

In [ ]:
strm_buf = gdf.dissolve()
strm_buf["geometry"] = strm_buf.geometry.buffer(75, cap_style=2, join_style=3)
strm_buf.plot();

The basin and stream polygons can now be used to generate a triangular mesh with the `Triangle` utility. The `Triangle` class has a number of input parameters. For more detail on the inputs and additional examples please see this [notebook](https://flopy.readthedocs.io/en/latest/Notebooks/dis_voronoi_example.html)

In [ ]:
# define point locations within the watershed and stream for adding regions
wsloc = [i[0] + 1000 for i in basindf.geometry[0].centroid.xy]
stloc = [i[0] + 2 for i in strm_buf.geometry[0].exterior.xy]

In [ ]:
tri = Triangle(angle=30, model_ws=tri_dir)

# define the model/mesh boundary
tri.add_polygon(basindf.geometry.values[0])
tri.add_region(wsloc, 0, maximum_area=100*300)

# define the stream refinement area
tri.add_polygon(strm_buf.geometry.values[0])
tri.add_region(stloc, 1, maximum_area=40*40)

In [ ]:
tri.build()

Visualize the triangular mesh

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
tri.plot(ax=ax)
gdf.plot(ax=ax);

#### Generating the Voronoi Mesh from the triangular mesh

A voronoi mesh can be generated from the circumcirle centroids of each triangular node in a Delaunay triangulation. The centroid of each circumcirle produces a vertex for the voronoi mesh and these vertices are connected via adjacency relationships of the triangulation.

FloPy has a built in utility `VoronoiGrid` that produces voronoi meshes from Delaunay triangulations. For more information and additional examples on `VoronoiGrid`'s usage, see this [notebook](https://flopy.readthedocs.io/en/latest/Notebooks/dis_voronoi_example.html)

To generate a voronoi mesh, we just need to pass our triangluation object (`tri`) to the `VoronoiGrid` utility

In [ ]:
vor = VoronoiGrid(tri)

Build a `VertexGrid`

In [ ]:
gridprops = vor.get_gridprops_vertexgrid()
nlay = 1
idomain = np.ones(gridprops["ncpl"], dtype=int)
top = idomain.copy().astype(float)
botm = np.zeros((nlay, gridprops["ncpl"]))

In [ ]:
vorgrid = VertexGrid(
    nlay=nlay,
    idomain=idomain,
    top=top,
    botm=botm,
    **gridprops
)

Resample the land surface elevation from the DEM to the voronoi grid

In [ ]:
top = rstr.resample_to_grid(
    vorgrid,
    band=rstr.bands[0],
    method="min",
    extrapolate_edges=True
)
vorgrid._top = top

Finally plot the voronoi mesh with the DEM data and stream locations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

pmv = PlotMapView(modelgrid=vorgrid)
pc = pmv.plot_array(vorgrid.top)
pmv.plot_inactive()
pmv.plot_grid(lw=0.3)
gdf.plot(ax=ax)
plt.colorbar(pc, shrink=0.7);

## Local Grid refinement (LGR style parent, child models)

A Local Grid Refinement (LGR) mesh can be created in an area of interest to get finer discretization than the existing/surrounding mesh. FloPy's `Lgr` utility, creates the refined mesh as a seperate modelgrid, and calculates the exchanges between the multi-model simulation mesh that results from refinement.

![LGR.png](../data/LGR.png)


The easiest way to build an locally refined models in a MF6 simulation structure is through FloPy's `Lgr` utility

The `Lgr` utility has a number of input parameters:
   - `nlayp`: the number of parent model layers
   - `nrowp`: the number of parent model rows
   - `ncolp`: the number of parent model columns
   - `delrp`: the parent model delr array
   - `delcp`: the parent model delc array
   - `topp`: the parent model top array
   - `botmp`: the parent model botm array
   - `idomainp`: an idomain array that is used to create the child grid. Values of 1 indicate parent model cells, values of 0 indicate child model cells. The child model must be a rectangular region that is continuous.
   - `ncpp`: number of child cells along the face of a parent cell
   - `ncppl`: number of child cells per parent layer
   - `xllp`: (optional) x location of parent grid lower left corner
   - `yllp`: (optional) y-location of parent grid lower left corner

In [ ]:
# load the refinement boundaries
lgr_poly = data_ws / "lgr_bound.shp"
lgr_gdf = gpd.read_file(lgr_poly)

Plot it to see if it lines up properly with our Structured discretization

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
pmv = PlotMapView(modelgrid=sgrid, ax=ax)
pc = pmv.plot_array(sgrid.top, cmap="viridis")
pmv.plot_inactive()
pmv.plot_grid(lw=0.5)
lgr_gdf.plot(ax=ax, facecolor="None", edgecolor="red", lw=4)
plt.colorbar(pc, shrink=0.7);

In [ ]:
lgr_gdf

Use GridIntersect to define the parent (1) and child (0) model areas with `idomainp`

In [ ]:
idomainp = np.ones(sgrid.shape, dtype=int)

ix = GridIntersect(sgrid)
rowcol = ix.intersect(lgr_gdf.geometry.values[0], contains_centroid=True)["cellids"]
crow, ccol = list(zip(*rowcol))
idomainp[:, crow, ccol] = 0

Create the Lgr object/child grid

In [ ]:
lgr = Lgr(
    nlayp=sgrid.nlay,
    nrowp=sgrid.nrow,
    ncolp=sgrid.ncol,
    delrp=sgrid.delr,
    delcp=sgrid.delc,
    topp=sgrid.top,
    botmp=sgrid.botm,
    idomainp=idomainp,
    ncpp=4,
    xllp=sgrid.xoffset,
    yllp=sgrid.yoffset
)

Get a `StructuredGrid` of the child model

In [ ]:
childgrid = lgr.child
childgrid

In [ ]:
parentgrid = lgr.parent
parentgrid

Resample top elevations to the child grid

In [ ]:
childtop = rstr.resample_to_grid(
    childgrid,
    band=rstr.bands[0],
    method="min"
)
childgrid._top = childtop

Do the same with the parent grid

In [ ]:
parenttop = rstr.resample_to_grid(
    parentgrid,
    band=rstr.bands[0],
    method="min"
)
parentgrid._top = parenttop

In [ ]:
# and remake the idomain array for the parent
ix = GridIntersect(sgrid)
result = ix.intersect(basindf.geometry.values[0], contains_centroid=True)
rowcol = result["cellids"]
idomain = parentgrid.idomain - 1
for row, col in rowcol:
    if idomain[0, row, col] > -1:
        idomain[0, row, col] = 1
    else:
        idomain[0, row, col] = 0

parentgrid._idomain = idomain

And now we can visualie the new LGR mesh

In [ ]:
fig, axs = plt.subplots(ncols=3, figsize=(12, 5))
vmin, vmax = np.min(sgrid.top), np.max(sgrid.top)

pmvp = PlotMapView(modelgrid=parentgrid, ax=axs[0])
pmvp.plot_array(parentgrid.top, cmap="viridis", vmin=vmin, vmax=vmax)
pmvp.plot_inactive()
pmvp.plot_grid(lw=0.3)
gdf.plot(ax=axs[0])

pmvc = PlotMapView(modelgrid=childgrid, ax=axs[1], extent=pmvp.extent)
pmvc.plot_array(childgrid.top, cmap="viridis", vmin=vmin, vmax=vmax)
pmvc.plot_inactive()
pmvc.plot_grid(lw=0.2)
gdf.plot(ax=axs[1])


pmvp = PlotMapView(modelgrid=parentgrid, ax=axs[2])
pmvp.plot_array(parentgrid.top, cmap="viridis", vmin=vmin, vmax=vmax)
pmvp.plot_inactive()
gdf.plot(ax=axs[2])

pmvc = PlotMapView(modelgrid=childgrid, ax=axs[2], extent=pmvp.extent)
pmvc.plot_array(childgrid.top, cmap="viridis", vmin=vmin, vmax=vmax)
pmvc.plot_grid(lw=0.2);

### More information
For more inforation and and examples on how to use the utilities presented in this notebook please see:

#### Examples
   - [FloPy Examples Gallery](https://flopy.readthedocs.io/en/latest/examples.html)

#### API reference
   - [FloPy utils API reference](https://flopy.readthedocs.io/en/latest/code.html#flopy-utilities)
   - [FloPy lgrutil API reference](https://flopy.readthedocs.io/en/latest/source/flopy.utils.lgrutil.html)
   - [FloPy gridgen API reference](https://flopy.readthedocs.io/en/latest/source/flopy.utils.gridgen.html)
   - [FloPy triangle API reference](https://flopy.readthedocs.io/en/latest/source/flopy.utils.triangle.html)
   - [FloPy voronoi API reference](https://flopy.readthedocs.io/en/latest/source/flopy.utils.voronoi.html)